In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('adult.csv')
df.head(5)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [3]:
df.shape

(48842, 15)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48842 non-null  int64 
 1   workclass        48842 non-null  object
 2   fnlwgt           48842 non-null  int64 
 3   education        48842 non-null  object
 4   educational-num  48842 non-null  int64 
 5   marital-status   48842 non-null  object
 6   occupation       48842 non-null  object
 7   relationship     48842 non-null  object
 8   race             48842 non-null  object
 9   gender           48842 non-null  object
 10  capital-gain     48842 non-null  int64 
 11  capital-loss     48842 non-null  int64 
 12  hours-per-week   48842 non-null  int64 
 13  native-country   48842 non-null  object
 14  income           48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


In [5]:
df.isnull().sum()

,0
age,0
workclass,0
fnlwgt,0
education,0
educational-num,0
marital-status,0
occupation,0
relationship,0
race,0
gender,0


In [6]:
df.describe()

,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


# Data Preprocessing :- Handling ? values

In [7]:
df_missing = (df=='?').sum()
df_missing

,0
age,0
workclass,2799
fnlwgt,0
education,0
educational-num,0
marital-status,0
occupation,2809
relationship,0
race,0
gender,0


In [8]:
df = df[df['workclass'] !='?']
df = df[df['occupation'] !='?']
df = df[df['native-country'] !='?']

In [9]:
df_missing = (df=='?').sum()
df_missing

,0
age,0
workclass,0
fnlwgt,0
education,0
educational-num,0
marital-status,0
occupation,0
relationship,0
race,0
gender,0


# Data Preprocessing :- Label encoding categorical values

In [10]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df_cat = df.select_dtypes(include=['object'])
df_cat = df_cat.apply(le.fit_transform)
df_cat.head(3)

,workclass,education,marital-status,occupation,relationship,race,gender,native-country,income
0,2,1,4,6,3,2,1,38,0
1,2,11,2,4,0,4,1,38,0
2,1,7,2,10,0,4,1,38,1


In [11]:
df = df.drop(df_cat.columns,axis=1)
df = pd.concat([df,df_cat],axis=1)
df.head()

,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week,workclass,education,marital-status,occupation,relationship,race,gender,native-country,income
0,25,226802,7,0,0,40,2,1,4,6,3,2,1,38,0
1,38,89814,9,0,0,50,2,11,2,4,0,4,1,38,0
2,28,336951,12,0,0,40,1,7,2,10,0,4,1,38,1
3,44,160323,10,7688,0,40,2,15,2,6,0,2,1,38,1
5,34,198693,6,0,0,30,2,0,4,7,1,4,1,38,0


# Test Train Split

In [12]:
from sklearn.model_selection import train_test_split

In [13]:
x = df.drop('income', axis=1)
y = df['income']

In [14]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.30)

# Applying various Boosting Techniques

## AdaBoost

In [15]:
from sklearn.ensemble import AdaBoostClassifier
model1 = AdaBoostClassifier(n_estimators=300, learning_rate=1)
model1.fit(x_train, y_train)

AdaBoostClassifier(learning_rate=1, n_estimators=300)

In [16]:
from sklearn.metrics import accuracy_score, confusion_matrix
x_test_pred = model1.predict(x_test)
cm = pd.DataFrame(confusion_matrix(y_test, x_test_pred), columns=['Actual -ve', 'Actual +ve'], index=['Predicted -ve', 'Predicted +ve'])
cm

,Actual -ve,Actual +ve
Predicted -ve,9599,608
Predicted +ve,1314,2046


In [17]:
from sklearn.metrics import classification_report
print(classification_report(y_test, x_test_pred))

              precision    recall  f1-score   support

           0       0.88      0.94      0.91     10207
           1       0.77      0.61      0.68      3360

    accuracy                           0.86     13567
   macro avg       0.83      0.77      0.79     13567
weighted avg       0.85      0.86      0.85     13567



### GradientBoost

In [18]:
from sklearn.ensemble import GradientBoostingClassifier
model2 = GradientBoostingClassifier(n_estimators=500, learning_rate=1, max_depth=5, subsample=0.9, min_samples_split=100, max_features='sqrt', random_state=10)
model2.fit(x_train, y_train)

GradientBoostingClassifier(learning_rate=1, max_depth=5, max_features='sqrt',
                           min_samples_split=100, n_estimators=500,
                           random_state=10, subsample=0.9)

In [19]:
x_test_pred = model2.predict(x_test)
cm = pd.DataFrame(confusion_matrix(y_test, x_test_pred), columns=['Actual -ve', 'Actual +ve'], index=['Predicted -ve', 'Predicted +ve'])
cm

,Actual -ve,Actual +ve
Predicted -ve,9083,1124
Predicted +ve,1659,1701


In [20]:
print(classification_report(y_test, x_test_pred))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87     10207
           1       0.60      0.51      0.55      3360

    accuracy                           0.79     13567
   macro avg       0.72      0.70      0.71     13567
weighted avg       0.79      0.79      0.79     13567



### XGBoost

In [21]:
import xgboost as xgb
from xgboost import XGBClassifier
model3 = XGBClassifier(n_estimators=1000, learning_rate=0.01, max_depth=20, colsample_bytree=0.4, gamma=1)
model3.fit(x_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.4, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=1, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.01, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=20,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=None, num_parallel_tree=None, ...)

In [22]:
x_test_pred = model3.predict(x_test)
cm = pd.DataFrame(confusion_matrix(y_test, x_test_pred), columns=['Actual -ve', 'Actual +ve'], index=['Predicted -ve', 'Predicted +ve'])
cm

,Actual -ve,Actual +ve
Predicted -ve,9642,565
Predicted +ve,1150,2210


In [23]:
print(classification_report(y_test, x_test_pred))

              precision    recall  f1-score   support

           0       0.89      0.94      0.92     10207
           1       0.80      0.66      0.72      3360

    accuracy                           0.87     13567
   macro avg       0.84      0.80      0.82     13567
weighted avg       0.87      0.87      0.87     13567

